---
title: Take-Home Final - Kaggle Regression
author: Ulises Caro
format: 
    html:
        code-fold: true
        embed-resources: true
echo: true
---

In [ ]:
import sklearn
import pandas as pd
import numpy as np
from sklearn.linear_model import *
from sklearn.preprocessing import *
from sklearn.metrics import *
from sklearn.model_selection import *
from sklearn.pipeline import Pipeline
from sklearn.compose import *
from sklearn.impute import SimpleImputer
import warnings
from sklearn.exceptions import ConvergenceWarning
from sklearn.neighbors import KNeighborsRegressor, KNeighborsClassifier
from sklearn.tree import DecisionTreeRegressor, DecisionTreeClassifier
from sklearn.linear_model import LinearRegression, LogisticRegression
from plotnine import *
from IPython.display import Markdown, display

In [2]:
train_data = "C:/Users/Ulises Caro Flores/OneDrive/Documents/MSBA Cal Poly/GSB 544 Computing and Machine Learning for Analysis/Finals/Kaggle Regression/train_new.csv"
test_data = "C:/Users/Ulises Caro Flores/OneDrive/Documents/MSBA Cal Poly/GSB 544 Computing and Machine Learning for Analysis/Finals/Kaggle Regression/test_new.csv"

train = pd.read_csv(train_data)
test = pd.read_csv(test_data)

In [3]:
print(train.dtypes)
print(train.describe())
print(train.nunique)

SalePrice          int64
PID                int64
Lot Frontage     float64
Lot Area           int64
Street            object
Neighborhood      object
Bldg Type         object
House Style       object
Overall Qual       int64
Overall Cond       int64
Year Built         int64
Roof Style        object
Heating           object
Central Air       object
Electrical        object
Full Bath          int64
Half Bath          int64
Bedroom AbvGr      int64
TotRms AbvGrd      int64
Gr Liv Area        int64
Functional        object
Screen Porch       int64
Pool Area          int64
Yr Sold            int64
Sale Type         object
dtype: object
           SalePrice           PID  Lot Frontage       Lot Area  Overall Qual  \
count    2197.000000  2.197000e+03   1835.000000    2197.000000   2197.000000   
mean   182376.851161  7.127872e+08     69.549319   10268.070096      6.116523   
std     81168.157405  1.887751e+08     23.188224    7975.740885      1.429613   
min     13100.000000  5.263011e+08   

In [4]:
train.isnull().sum()

SalePrice          0
PID                0
Lot Frontage     362
Lot Area           0
Street             0
Neighborhood       0
Bldg Type          0
House Style        0
Overall Qual       0
Overall Cond       0
Year Built         0
Roof Style         0
Heating            0
Central Air        0
Electrical         1
Full Bath          0
Half Bath          0
Bedroom AbvGr      0
TotRms AbvGrd      0
Gr Liv Area        0
Functional         0
Screen Porch       0
Pool Area          0
Yr Sold            0
Sale Type          0
dtype: int64

In [5]:
test.isnull().sum()

PID              0
Lot Frontage     0
Lot Area         0
Street           0
Neighborhood     0
Bldg Type        0
House Style      0
Overall Qual     0
Overall Cond     0
Year Built       0
Roof Style       0
Heating          0
Central Air      0
Electrical       0
Full Bath        0
Half Bath        0
Bedroom AbvGr    0
TotRms AbvGrd    0
Gr Liv Area      0
Functional       0
Screen Porch     0
Pool Area        0
Yr Sold          0
Sale Type        0
dtype: int64

In [6]:
print(test.shape[0])
print(train.shape[0])

605
2197


In [7]:
X = train.drop(columns=["SalePrice","PID"])
y = np.log(train["SalePrice"])

In [8]:
num_var = X.select_dtypes(include=["int64", "float64"]).columns.tolist()
cat_var = X.select_dtypes(include=["object", "category"]).columns.tolist()

num_pipe = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

cat_pipe = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

ct = ColumnTransformer(
    transformers=[
        ("num", num_pipe, num_var),
        ("cat", cat_pipe, cat_var),
    ]
)

score = "neg_root_mean_squared_error"
CV = 10

**Will comparing the models based on the best RMSE and hypertuning for: Linear, Lasso, Ridge, Elastic Net, KNN, & DT**

In [9]:
#Linear Regression
lr_pipe = Pipeline([
    ("preprocess", ct),
    ("lr", LinearRegression())
])

lr_pipe.fit(X, y)

lr_cv = cross_val_score(lr_pipe, X, y, cv=CV, scoring=score)
lr_est = -lr_cv.mean()
print(f"Estimated RMSE Linear Regression: {lr_est:.4f}")

Estimated RMSE Linear Regression: 0.1487


In [10]:
#Lasso Regression
lasso_pipe = Pipeline([
    ("preprocess", ct),
    ("lasso", Lasso())
])

lasso_pipe.fit(X, y)

para_lasso = {
    "lasso__alpha": [0.0001, 0.001, 0.01, 0.1, 1, 10, 100],
    "lasso__max_iter": [10000],
}

lasso_search = GridSearchCV(lasso_pipe, para_lasso, cv=CV, scoring=score)
lasso_fit    = lasso_search.fit(X, y)

In [11]:
lasso_fit.best_estimator_.named_steps["lasso"]

,alpha,0.0001
,fit_intercept,True
,precompute,False
,copy_X,True
,max_iter,10000
,tol,0.0001
,warm_start,False
,positive,False
,random_state,None
,selection,'cyclic'


In [12]:
lasso_best = lasso_fit.best_estimator_
lasso_cv = cross_val_score(lasso_best, X, y, cv=CV, scoring=score)
lasso_est = -lasso_cv.mean()
print(f"Estimated RMSE Lasso: {lasso_est:.4f}")

Estimated RMSE Lasso: 0.1483


In [17]:
#Ridge Regression
ridge_pipe = Pipeline([
    ("preprocess", ct),
    ("ridge", Ridge())
])

ridge_pipe.fit(X, y)

para_ridge = {
    "ridge__alpha": [0.01, 0.1, 1, 10, 100],
    "ridge__max_iter": [10000],
}

ridge_search = GridSearchCV(ridge_pipe, para_ridge, cv=CV, scoring=score)
ridge_fit    = ridge_search.fit(X, y)

In [18]:
ridge_fit.best_estimator_.named_steps["ridge"]

,alpha,10
,fit_intercept,True
,copy_X,True
,max_iter,10000
,tol,0.0001
,solver,'auto'
,positive,False
,random_state,None


In [19]:
ridge_best = ridge_fit.best_estimator_
ridge_cv = cross_val_score(ridge_best, X, y, cv=CV, scoring=score)
ridge_est = -ridge_cv.mean()
print(f"Estimated RMSE Ridge: {ridge_est:.4f}")

Estimated RMSE Ridge: 0.1483


In [20]:
#Elastic Net
net_pipe = Pipeline([
    ("preprctocess", ct),
    ("net", ElasticNet())
])

net_pipe.fit(X, y)

para_net = {
    "net__alpha": [0.0001, 0.001, 0.01, 0.1, 1, 10, 100],
    "net__l1_ratio": [0.2, 0.5, 0.8],
    "net__max_iter": [10000],
}

net_search = GridSearchCV(net_pipe, para_net, cv=CV, scoring=score)
net_fit = net_search.fit(X, y)

In [21]:
net_fit.best_estimator_.named_steps["net"]

,alpha,0.001
,l1_ratio,0.2
,fit_intercept,True
,precompute,False
,max_iter,10000
,copy_X,True
,tol,0.0001
,warm_start,False
,positive,False
,random_state,None
,selection,'cyclic'


In [22]:
net_best = net_fit.best_estimator_
net_cv = cross_val_score(net_best, X, y, cv=CV, scoring=score)
net_est = -net_cv.mean()
print(f"Estimated RMSE Elastic Net: {net_est:.4f}")

Estimated RMSE Elastic Net: 0.1482


In [23]:
#KNN Regression
knn_pipe = Pipeline([
    ("preprocess", ct),
    ("knn", KNeighborsRegressor())
])

knn_pipe.fit(X, y)

para_knn = {
    'knn__n_neighbors': range(1, 10+1),
    'knn__metric': ['manhattan', 'euclidean']
}

knn_search = GridSearchCV(knn_pipe, para_knn, cv=CV, scoring=score)
knn_fit    = knn_search.fit(X, y)

In [24]:
knn_fit.best_estimator_.named_steps["knn"]

,n_neighbors,7
,weights,'uniform'
,algorithm,'auto'
,leaf_size,30
,p,2
,metric,'manhattan'
,metric_params,None
,n_jobs,None


In [25]:
knn_best = knn_fit.best_estimator_
knn_cv = cross_val_score(knn_best, X, y, cv=CV, scoring=score)
knn_est = -knn_cv.mean()
print(f"Estimated RMSE KNN: {knn_est:.4f}")

Estimated RMSE KNN: 0.1658


In [26]:
#Decision Tree Regression
dt_pipe = Pipeline([
    ("preprocess", ct),
    ("dt", DecisionTreeRegressor())
])

dt_pipe.fit(X, y)

para_dt = {
    "dt__max_depth": [None, 3, 5, 8, 12],
    "dt__min_samples_split": [2, 5, 10],
    "dt__min_samples_leaf": [1, 2, 5],
}

dt_search = GridSearchCV(dt_pipe, para_dt, cv=CV, scoring=score)
dt_fit    = dt_search.fit(X, y)

In [27]:
dt_fit.best_estimator_.named_steps["dt"]

,criterion,'squared_error'
,splitter,'best'
,max_depth,8
,min_samples_split,5
,min_samples_leaf,5
,min_weight_fraction_leaf,0.0
,max_features,None
,random_state,None
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,ccp_alpha,0.0


In [28]:
dt_best = dt_fit.best_estimator_
dt_cv = cross_val_score(dt_best, X, y, cv=CV, scoring=score)
dt_est = -dt_cv.mean()
print(f"Estimated RMSE Decision Tree: {dt_est:.4f}")

Estimated RMSE Decision Tree: 0.1787


In [29]:
#Comparison to select the best model


results_df = pd.DataFrame({
        "RMSE": [lr_est,
            ridge_est,
            lasso_est,
            net_est,
            knn_est,
            dt_est
       ]},
    index=["Linear Regression",
        "Ridge",
        "Lasso",
        "Elastic Net",
        "KNN",
        "Decision Tree"])

results_df.sort_values(by="RMSE",ascending=True).style.format("{:.4f}")

,RMSE
Elastic Net,0.1482
Lasso,0.1483
Ridge,0.1483
Linear Regression,0.1487
KNN,0.1658
Decision Tree,0.1787


In [30]:
#Selecting Elastic Net and Predicting using the test data.

X_test = test.drop(columns=["PID"])

In [31]:
y_pred = net_best.predict(X_test)
predictions_exp = np.exp(y_pred)

submission_regression = pd.DataFrame({
    "PID": test["PID"],
    "SalePrice": predictions_exp
})
print(submission_regression.head())
print(submission_regression.shape[0])
print(test.shape[0])

         PID      SalePrice
0  907135180  127071.983899
1  528181040  219836.500271
2  528175010  219243.757805
3  531379030  186015.621908
4  923275090  129378.282938
605
605


In [33]:
submission_regression.to_csv("submission_regression.csv", index=False)
print("Submission file created!")

Submission file created!


**Refining Elastic Net model**

In [36]:
best_alpha = net_fit.best_params_["net__alpha"]
best_l1    = net_fit.best_params_["net__l1_ratio"]

print(f"The best alpha and best l1 for Elastic Net are: alpha = {best_alpha}, l1_ratio = {best_l1}")

The best alpha and best l1 for Elastic Net are: alpha = 0.001, l1_ratio = 0.2


In [39]:
para_net_refined = {
    "net__alpha": [
        best_alpha, 
        best_alpha * 1.5, 
        best_alpha * 2, 
        best_alpha * 3
    ],
    "net__l1_ratio": [
        max(0.0, best_l1 - 0.1),
        best_l1,
        min(1.0, best_l1 + 0.1)
    ],
    "net__max_iter": [50000]
}

net_search_refined = GridSearchCV(net_pipe, para_net_refined, cv=CV, scoring=score)
net_fit_refined    = net_search_refined.fit(X, y)

In [40]:
net_best_refined = net_fit_refined.best_estimator_
net_cv2 = cross_val_score(net_best_refined, X, y, cv=CV, scoring=score)
net_est2 = -net_cv2.mean()
print(f"Estimated RMSE for a Refined Elastic Net: {net_est2:.4f}")


Estimated RMSE for a Refined Elastic Net: 0.1482


**Refining the Elastic did nothing**

**Prediction using Lasso/Ridge on test set**

In [47]:
y_pred2 = lasso_best.predict(X_test)
predictions_exp2 = np.exp(y_pred2)

submission_regression_lasso = pd.DataFrame({
    "PID": test["PID"],
    "SalePrice": predictions_exp2
})

submission_regression_lasso.to_csv("submission_regression_lasso.csv", index=False)
print("Submission file created!")

Submission file created!


In [46]:
y_pred3 = ridge_best.predict(X_test)
predictions_exp3 = np.exp(y_pred3)

submission_regression_ridge = pd.DataFrame({
    "PID": test["PID"],
    "SalePrice": predictions_exp3
})

submission_regression_ridge.to_csv("submission_regression_ridge.csv", index=False)
print("Submission file created!")

Submission file created!


**Lasso (0.1540) performed much worse than both while Ridge (0.1492) slightly did worse**